In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('Causal_SocialComparison.csv')
df['Group'].value_counts()

Group
Treatment    70
Control      70
Name: count, dtype: int64

In [3]:
df.head()

,UserID,Group,weekly_hour,Pre_hour,Age
0,38908,Treatment,4.6907,50.52,22
1,39080,Treatment,4.4387,49.59,28
2,39206,Control,2.8347,36.10,28
3,39290,Treatment,4.5796,45.33,22
4,39370,Control,3.4767,40.63,39


In [4]:
group_stats = df.groupby('Group')[['weekly_hour', 'Pre_hour', 'Age']].agg(['mean', 'std'])
print(group_stats)

          weekly_hour             Pre_hour                   Age          
                 mean       std       mean        std       mean       std
Group                                                                     
Control      3.486636  0.345508  38.545857  19.954883  25.214286  4.201079
Treatment    4.564601  0.369740  41.689000  17.676531  26.157143  5.465665


# t-tests

## t-test on the average weekly-hours across the two groups

In [5]:
from scipy.stats import ttest_ind

# Extract weekly_hour values for each group
weekly_hour_treatment = df[df['Group'] == 'Treatment']['weekly_hour']
weekly_hour_control = df[df['Group'] == 'Control']['weekly_hour']

avg_weekly_hour_treatment = weekly_hour_treatment.mean()
avg_weekly_hour_control = weekly_hour_control.mean()
print(f"Average weekly_hour (Treatment): {avg_weekly_hour_treatment:.4f}")
print(f"Average weekly_hour (control): {avg_weekly_hour_control:.4f}")
print(f"Difference in averages: {avg_weekly_hour_treatment - avg_weekly_hour_control:.4f}")
# Perform independent t-test
t_stat, p_value = ttest_ind(weekly_hour_treatment, weekly_hour_control, equal_var=False)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

Average weekly_hour (Treatment): 4.5646
Average weekly_hour (control): 3.4866
Difference in averages: 1.0780
T-statistic: 17.8223
P-value: 0.0000


## t-test for Age

In [22]:

# Extract weekly_hour values for each group
Age_treatment = df[df['Group'] == 'Treatment']['Age']
Age_control = df[df['Group'] == 'Control']['Age']

avg_Age_treatment = Age_treatment.mean()
avg_Age_control = Age_control.mean()
print(f"Average Age (Treatment): {avg_Age_treatment:.4f}")
print(f"Average Age (control): {avg_Age_control:.4f}")

# Perform independent t-test
t_stat, p_value = ttest_ind(Age_treatment, Age_control, equal_var=False)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

Average Age (Treatment): 26.1571
Average Age (control): 25.2143
T-statistic: 1.1443
P-value: 0.2546


## t-test for pre-hour

In [23]:
Pre_hour_treatment = df[df['Group'] == 'Treatment']['Pre_hour']
Pre_hour_control = df[df['Group'] == 'Control']['Pre_hour']

avg_Pre_hour_treatment = Pre_hour_treatment.mean()
avg_Pre_hour_control = Pre_hour_control.mean()
print(f"Average Pre_hour (Treatment): {avg_Pre_hour_treatment:.4f}")
print(f"Average Pre_hour (control): {avg_Pre_hour_control:.4f}")

# Perform independent t-test
t_stat, p_value = ttest_ind(Pre_hour_treatment, Pre_hour_control, equal_var=False)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

Average Pre_hour (Treatment): 41.6890
Average Pre_hour (control): 38.5459
T-statistic: 0.9865
P-value: 0.3257


# Heterogeneous Treatment Effect - using regression

In [25]:
import statsmodels.formula.api as smf

# Create treatment dummy (1 if Treatment, 0 if Control)
df['treatment_dummy'] = (df['Group'] == 'Treatment').astype(int)

# Create age_above_30 dummy (1 if Age > 30, else 0)
df['age_above_30'] = (df['Age'] > 30).astype(int)

# Interaction term: treatment_dummy * age_above_30
df['treat_age30_interaction'] = df['treatment_dummy'] * df['age_above_30']

# Run regression
model = smf.ols('weekly_hour ~ treatment_dummy + Age + Pre_hour + age_above_30+treat_age30_interaction', data=df).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            weekly_hour   R-squared:                       0.725
Model:                            OLS   Adj. R-squared:                  0.714
Method:                 Least Squares   F-statistic:                     70.56
Date:                Sat, 31 May 2025   Prob (F-statistic):           7.65e-36
Time:                        10:31:24   Log-Likelihood:                -47.073
No. Observations:                 140   AIC:                             106.1
Df Residuals:                     134   BIC:                             123.8
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 